In [1]:
# ============================================================
# CELL 1: INSTALL + IMPORT + CONFIG
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets scipy scikit-learn

# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import csv
import json
import time
import math
import random
import logging
import re

from pathlib import Path
from collections import Counter

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch.optim import AdamW

from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence
)

from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from scipy.stats import (
    pearsonr,
    spearmanr
)

# ============================================================
# OUTPUT
# ============================================================

OUTPUT_ROOT = "/content/drive/MyDrive/02_STL_BILSTM"

Path(OUTPUT_ROOT).mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# HYPERPARAMETERS
# ============================================================

HP = dict(

    embed_dim=300,

    hidden=256,

    num_layers=2,

    dropout=0.5,

    vocab_size=50000,

    max_seq_length=128,

    batch_size=32,

    eval_batch_size=64,

    learning_rate=3e-4,

    weight_decay=1e-5,

    seed=42,

    grad_clip=1.0,

    patience=3,

    max_epochs=15,

    num_workers=2,

    bench_max_batches=50,
)

# ============================================================
# TASKS
# ============================================================

TASKS = {

    "sst2": {
        "hf": ("glue", "sst2"),
        "type": "cls",
        "a": "sentence",
        "b": None,
        "num_labels": 2,
        "primary": "accuracy",
    },

    "qqp": {
        "hf": ("glue", "qqp"),
        "type": "cls",
        "a": "question1",
        "b": "question2",
        "num_labels": 2,
        "primary": "accuracy",
    },

    "stsb": {
        "hf": ("glue", "stsb"),
        "type": "reg",
        "a": "sentence1",
        "b": "sentence2",
        "num_labels": 1,
        "primary": "pearson",
    },
}

# ============================================================
# EXPERIMENTS
# ============================================================

EXPERIMENTS = [

    {
        "exp_name": "BiLSTM-SST2",
        "task": "sst2",
    },

    {
        "exp_name": "BiLSTM-QQP",
        "task": "qqp",
    },

    {
        "exp_name": "BiLSTM-STSB",
        "task": "stsb",
    },
]

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("DEVICE =", DEVICE)

# ============================================================
# SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# LOGGER
# ============================================================

def make_logger(name, log_path):

    lg = logging.getLogger(name)

    lg.setLevel(logging.INFO)

    lg.propagate = False

    for h in list(lg.handlers):
        lg.removeHandler(h)

    fmt = logging.Formatter(
        "[%(asctime)s] [%(levelname)s] %(message)s",
        "%Y-%m-%d %H:%M:%S"
    )

    fh = logging.FileHandler(
        log_path,
        mode="w",
        encoding="utf-8"
    )

    fh.setFormatter(fmt)

    sh = logging.StreamHandler(sys.stdout)

    sh.setFormatter(fmt)

    lg.addHandler(fh)
    lg.addHandler(sh)

    return lg

# ============================================================
# HISTORY WRITER
# ============================================================

class HistoryWriter:

    def __init__(self, out_dir):

        self.csv_path = Path(out_dir) / "history.csv"

        self.records = []

        self.fields = []

    def append(self, rec):

        for k in rec.keys():

            if k not in self.fields:
                self.fields.append(k)

        self.records.append(rec)

        with open(
            self.csv_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=self.fields
            )

            writer.writeheader()

            for r in self.records:
                writer.writerow(r)

# ============================================================
# TOKENIZER
# ============================================================

TOK_RE = re.compile(
    r"[A-Za-z0-9']+|[^\sA-Za-z0-9']"
)

def tokenize(text):

    return TOK_RE.findall(
        text.lower()
    )

# ============================================================
# VOCAB
# ============================================================

class Vocab:

    PAD = "<pad>"

    UNK = "<unk>"

    def __init__(self):

        self.itos = [
            self.PAD,
            self.UNK
        ]

        self.stoi = {
            self.PAD: 0,
            self.UNK: 1
        }

    def build(self, corpus, max_size):

        counter = Counter()

        for s in corpus:

            counter.update(
                tokenize(s)
            )

        for tok, _ in counter.most_common(
            max_size - len(self.itos)
        ):

            self.stoi[tok] = len(self.itos)

            self.itos.append(tok)

    def encode(self, text, max_len):

        ids = [

            self.stoi.get(t, 1)

            for t in tokenize(text)

        ][:max_len]

        return ids

    def __len__(self):

        return len(self.itos)

# ============================================================
# DATASET
# ============================================================

class BiLSTMDataset(Dataset):

    def __init__(
        self,
        hf_split,
        task,
        vocab,
        max_len
    ):

        m = TASKS[task]

        self.a = list(
            hf_split[m["a"]]
        )

        self.b = (

            list(hf_split[m["b"]])

            if m["b"] else None
        )

        self.y = np.asarray(
            hf_split["label"],
            dtype=np.float32
            if m["type"] == "reg"
            else np.int64
        )

        self.vocab = vocab

        self.max_len = max_len

        self.is_pair = (
            m["b"] is not None
        )

    def __len__(self):

        return len(self.a)

    def __getitem__(self, i):

        a_ids = self.vocab.encode(
            self.a[i],
            self.max_len
        )

        if len(a_ids) == 0:
            a_ids = [1]

        if self.is_pair:

            b_ids = self.vocab.encode(
                self.b[i],
                self.max_len
            )

            if len(b_ids) == 0:
                b_ids = [1]

        else:

            b_ids = None

        return {

            "a": a_ids,

            "b": b_ids,

            "y": self.y[i]
        }

# ============================================================
# COLLATE
# ============================================================

def collate(batch, is_pair, is_reg):

    a = [

        torch.tensor(
            b["a"],
            dtype=torch.long
        )

        for b in batch
    ]

    la = torch.tensor(
        [len(x) for x in a]
    )

    a = pad_sequence(
        a,
        batch_first=True,
        padding_value=0
    )

    out = {
        "a": a,
        "la": la
    }

    if is_pair:

        bb = [

            torch.tensor(
                b["b"],
                dtype=torch.long
            )

            for b in batch
        ]

        lb = torch.tensor(
            [len(x) for x in bb]
        )

        bb = pad_sequence(
            bb,
            batch_first=True,
            padding_value=0
        )

        out["b"] = bb

        out["lb"] = lb

    y = np.array([
        b["y"]
        for b in batch
    ])

    out["y"] = torch.tensor(
        y,
        dtype=torch.float32
        if is_reg
        else torch.long
    )

    return out

Mounted at /content/drive
DEVICE = cuda


In [ ]:
# ============================================================
# CELL 2: MODEL + TRAIN + FINAL SUMMARY + TIME
# ============================================================

# ============================================================
# MODEL
# ============================================================

class BiLSTMEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden,
        num_layers,
        dropout
    ):

        super().__init__()

        self.emb = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.out_dim = hidden * 2

    def forward(self, ids, lens):

        x = self.dropout(
            self.emb(ids)
        )

        packed = pack_padded_sequence(
            x,
            lens.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (h, _) = self.lstm(packed)

        h = h.view(
            self.lstm.num_layers,
            2,
            ids.size(0),
            self.lstm.hidden_size
        )

        h_last = h[-1]

        rep = torch.cat(
            [
                h_last[0],
                h_last[1]
            ],
            dim=-1
        )

        return self.dropout(rep)

# ============================================================
# MAIN MODEL
# ============================================================

class BiLSTMModel(nn.Module):

    def __init__(self, vocab_size, task):

        super().__init__()

        self.task = task

        self.is_pair = (
            TASKS[task]["b"] is not None
        )

        self.is_reg = (
            TASKS[task]["type"] == "reg"
        )

        self.enc = BiLSTMEncoder(
            vocab_size,
            HP["embed_dim"],
            HP["hidden"],
            HP["num_layers"],
            HP["dropout"]
        )

        h = self.enc.out_dim

        feat_dim = (
            h * 4
            if self.is_pair
            else h
        )

        out_dim = (
            1
            if self.is_reg
            else TASKS[task]["num_labels"]
        )

        self.head = nn.Sequential(

            nn.Linear(feat_dim, h),

            nn.ReLU(),

            nn.Dropout(HP["dropout"]),

            nn.Linear(h, out_dim)
        )

        self.loss_cls = nn.CrossEntropyLoss()

        self.loss_reg = nn.MSELoss()

    def forward(
        self,
        a,
        la,
        b=None,
        lb=None,
        y=None
    ):

        ha = self.enc(a, la)

        if self.is_pair:

            hb = self.enc(b, lb)

            feat = torch.cat(
                [
                    ha,
                    hb,
                    torch.abs(ha - hb),
                    ha * hb
                ],
                dim=-1
            )

        else:

            feat = ha

        z = self.head(feat)

        if self.is_reg:
            z = z.squeeze(-1)

        if y is None:
            return None, z

        if self.is_reg:

            loss = self.loss_reg(
                z,
                y
            )

        else:

            loss = self.loss_cls(
                z,
                y
            )

        return loss, z

# ============================================================
# EVALUATE
# ============================================================

@torch.no_grad()
def evaluate(model, loader, task):

    model.eval()

    preds = []

    labels = []

    losses = []

    for batch in loader:

        bb = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        y = bb.pop("y")

        loss, z = model(**bb, y=y)

        losses.append(loss.item())

        if TASKS[task]["type"] == "cls":

            p = z.argmax(-1)

        else:

            p = z

        preds.append(
            p.cpu().numpy()
        )

        labels.append(
            y.cpu().numpy()
        )

    y_pred = np.concatenate(preds)

    y_true = np.concatenate(labels)

    if TASKS[task]["type"] == "cls":

        return {

            "accuracy":
                accuracy_score(
                    y_true,
                    y_pred
                ),

            "precision":
                precision_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                ),

            "recall":
                recall_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                ),

            "macro_f1":
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                ),

            "pearson": np.nan,

            "spearman": np.nan,

            "eval_loss":
                np.mean(losses),
        }

    else:

        return {

            "accuracy": np.nan,

            "precision": np.nan,

            "recall": np.nan,

            "macro_f1": np.nan,

            "pearson":
                pearsonr(
                    y_pred,
                    y_true
                )[0],

            "spearman":
                spearmanr(
                    y_pred,
                    y_true
                )[0],

            "eval_loss":
                np.mean(losses),
        }

# ============================================================
# BENCHMARK
# ============================================================

@torch.no_grad()
def benchmark(model, loader):

    model.eval()

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats()

        torch.cuda.synchronize()

    t0 = time.perf_counter()

    n = 0

    for i, batch in enumerate(loader):

        if i >= HP["bench_max_batches"]:
            break

        bb = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "y"
        }

        _ = model(**bb)

        n += bb["a"].shape[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    el = time.perf_counter() - t0

    vram = (

        torch.cuda.max_memory_allocated()
        / 1024**2

        if torch.cuda.is_available()

        else 0.0
    )

    return {

        "inference_latency_ms":
            (el / n) * 1000,

        "throughput_samples_per_sec":
            n / el,

        "peak_vram_mb":
            vram,
    }

# ============================================================
# RUN ONE
# ============================================================

def run_one(exp):

    task = exp["task"]

    meta = TASKS[task]

    out_dir = (
        Path(OUTPUT_ROOT)
        / exp["exp_name"]
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    logger = make_logger(
        exp["exp_name"],
        str(out_dir / "train.log")
    )

    logger.info(
        f"===== {exp['exp_name']} ====="
    )

    set_seed(HP["seed"])

    total_train_start = time.perf_counter()

    raw = load_dataset(*meta["hf"])

    corpus = list(
        raw["train"][meta["a"]]
    )

    if meta["b"]:

        corpus += list(
            raw["train"][meta["b"]]
        )

    vocab = Vocab()

    vocab.build(
        corpus,
        HP["vocab_size"]
    )

    train_ds = BiLSTMDataset(
        raw["train"],
        task,
        vocab,
        HP["max_seq_length"]
    )

    val_ds = BiLSTMDataset(
        raw["validation"],
        task,
        vocab,
        HP["max_seq_length"]
    )

    coll = lambda b: collate(
        b,
        meta["b"] is not None,
        meta["type"] == "reg"
    )

    train_dl = DataLoader(
        train_ds,
        batch_size=HP["batch_size"],
        shuffle=True,
        num_workers=HP["num_workers"],
        collate_fn=coll
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=HP["eval_batch_size"],
        shuffle=False,
        num_workers=HP["num_workers"],
        collate_fn=coll
    )

    model = BiLSTMModel(
        len(vocab),
        task
    ).to(DEVICE)

    opt = AdamW(
        model.parameters(),
        lr=HP["learning_rate"],
        weight_decay=HP["weight_decay"],
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            opt,
            mode="max",
            factor=0.5,
            patience=1
        )
    )

    hist = HistoryWriter(out_dir)

    best = -1e9

    wait = 0

    # ========================================================
    # TRAIN LOOP
    # ========================================================

    for ep in range(
        1,
        HP["max_epochs"] + 1
    ):

        epoch_start = time.perf_counter()

        model.train()

        losses = []

        for batch in train_dl:

            bb = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            y = bb.pop("y")

            opt.zero_grad()

            loss, _ = model(
                **bb,
                y=y
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                HP["grad_clip"]
            )

            opt.step()

            losses.append(
                loss.item()
            )

        epoch_time = (
            time.perf_counter()
            - epoch_start
        )

        train_loss = np.mean(losses)

        metrics = evaluate(
            model,
            val_dl,
            task
        )

        score = metrics[
            meta["primary"]
        ]

        scheduler.step(score)

        bench = benchmark(
            model,
            val_dl
        )

        row = {

            "epoch": ep,

            "train_loss":
                train_loss,

            "eval_loss":
                metrics["eval_loss"],

            "accuracy":
                metrics["accuracy"],

            "precision":
                metrics["precision"],

            "recall":
                metrics["recall"],

            "macro_f1":
                metrics["macro_f1"],

            "pearson":
                metrics["pearson"],

            "spearman":
                metrics["spearman"],

            "inference_latency_ms":
                bench["inference_latency_ms"],

            "throughput_samples_per_sec":
                bench["throughput_samples_per_sec"],

            "peak_vram_mb":
                bench["peak_vram_mb"],

            "time_per_epoch":
                epoch_time,
        }

        hist.append(row)

        logger.info(row)

        if score > best:

            best = score

            wait = 0

            torch.save(
                model.state_dict(),
                out_dir / "best_model.pt"
            )

            logger.info(
                "NEW BEST"
            )

        else:

            wait += 1

            logger.info(
                f"No improvement "
                f"{wait}/{HP['patience']}"
            )

        if wait >= HP["patience"]:

            logger.info(
                "EARLY STOPPING"
            )

            break

    # ========================================================
    # TOTAL TRAIN TIME
    # ========================================================

    total_training_time = (
        time.perf_counter()
        - total_train_start
    )

    result = {

        "exp_name":
            exp["exp_name"],

        "task":
            task,

        "best_score":
            best,

        "accuracy":
            metrics["accuracy"],

        "precision":
            metrics["precision"],

        "recall":
            metrics["recall"],

        "macro_f1":
            metrics["macro_f1"],

        "pearson":
            metrics["pearson"],

        "spearman":
            metrics["spearman"],

        "train_loss":
            train_loss,

        "eval_loss":
            metrics["eval_loss"],

        "inference_latency_ms":
            bench["inference_latency_ms"],

        "throughput_samples_per_sec":
            bench["throughput_samples_per_sec"],

        "peak_vram_mb":
            bench["peak_vram_mb"],

        "epochs_ran":
            ep,

        "total_training_time":
            total_training_time,
    }

    with open(
        out_dir / "summary.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            indent=2,
            default=str
        )

    with open(
        out_dir / "summary.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=result.keys()
        )

        writer.writeheader()

        writer.writerow(result)

    return result

# ============================================================
# RUN ALL
# ============================================================

def run_all():

    final_results = []

    for exp in EXPERIMENTS:

        r = run_one(exp)

        final_results.append(r)

    final_csv = (
        Path(OUTPUT_ROOT)
        / "final_summary.csv"
    )

    with open(
        final_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=final_results[0].keys()
        )

        writer.writeheader()

        for r in final_results:
            writer.writerow(r)

    with open(
        Path(OUTPUT_ROOT)
        / "final_summary.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            final_results,
            f,
            indent=2,
            default=str
        )

    print("\n===== FINAL RESULTS =====")

    for r in final_results:

        print(
            f"{r['exp_name']} | "
            f"acc={r['accuracy']} | "
            f"f1={r['macro_f1']} | "
            f"pearson={r['pearson']} | "
            f"time={r['total_training_time']:.2f}s"
        )

# ============================================================
# MAIN
# ============================================================

run_all()

[2026-05-18 04:01:35] [INFO] ===== BiLSTM-SST2 =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

...

qqp/test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

[2026-05-18 04:14:02] [INFO] {'epoch': 1, 'train_loss': np.float64(0.5168711313242838), 'eval_loss': np.float64(0.4699900640717036), 'accuracy': 0.7647786297303982, 'precision': 0.7506324410468527, 'recall': 0.7632825367683196, 'macro_f1': 0.7542563323021962, 'pearson': nan, 'spearman': nan, 'inference_latency_ms': 0.2144176015625021, 'throughput_samples_per_sec': 4663.796221545286, 'peak_vram_mb': 387.1845703125, 'time_per_epoch': 359.39237449199993}
[2026-05-18 04:14:03] [INFO] NEW BEST
[2026-05-18 04:20:13] [INFO] {'epoch': 2, 'train_loss': np.float64(0.4653135223042083), 'eval_loss': np.float64(0.510079499664186), 'accuracy': 0.7483551817956963, 'precision': 0.753002533762317, 'recall': 0.7716065749200187, 'macro_f1': 0.7449670472368716, 'pearson': nan, 'spearman': nan, 'inference_latency_ms': 0.20502934906247106, 'throughput_samples_per_sec': 4877.3505089522905, 'peak_vram_mb': 387.1845703125, 'time_per_epoch': 361.4749629789999}
[2026-05-18 04:20:13] [INFO] No improvement 1/3
[20